In [116]:
import numpy as np
import pandas as pd
import geopandas as gpd

In [117]:
df = pd.read_csv("../Week3/version2/cleaned_housing_data.csv")

In [118]:
districts = gpd.read_file(
    "DistrictAreas2425/DistrictAreas2425.shp"
)

print(districts.shape)
print(districts.columns)
districts.head()

(937, 54)
Index(['OBJECTID', 'Year', 'FedID', 'CDCode', 'CDSCode', 'CountyName',
       'DistrictNa', 'DistrictTy', 'GradeLow', 'GradeHigh', 'GradeLowCe',
       'GradeHighC', 'AssistStat', 'CongressUS', 'SenateCA', 'AssemblyCA',
       'UpdateNote', 'EnrollTota', 'EnrollChar', 'EnrollNonC', 'AAcount',
       'AApct', 'AIcount', 'AIpct', 'AScount', 'ASpct', 'FIcount', 'FIpct',
       'HIcount', 'HIpct', 'PIcount', 'PIpct', 'WHcount', 'WHpct', 'MRcount',
       'MRpct', 'NRcount', 'NRpct', 'ELcount', 'ELpct', 'FOScount', 'FOSpct',
       'HOMcount', 'HOMpct', 'MIGcount', 'MIGpct', 'SWDcount', 'SWDpct',
       'SEDcount', 'SEDpct', 'DistrctAre', 'Shape__Are', 'Shape__Len',
       'geometry'],
      dtype='str')


,OBJECTID,Year,FedID,CDCode,CDSCode,CountyName,DistrictNa,DistrictTy,GradeLow,GradeHigh,...,MIGcount,MIGpct,SWDcount,SWDpct,SEDcount,SEDpct,DistrctAre,Shape__Are,Shape__Len,geometry
0,1,2024-25,0601770,0161119,01611190000000,Alameda,Alameda Unified,Unified,PK,12,...,0,0.0,1356,12.6,3958,36.7,11.248939,4.755489e+07,56522.982683,"MULTIPOLYGON (((-13606222.82 4540862.699, -136..."
1,2,2024-25,0601860,0161127,01611270000000,Alameda,Albany City Unified,Unified,PK,12,...,0,0.0,357,9.7,1184,32.1,1.789984,7.096327e+06,12696.382797,"POLYGON ((-13612893.866 4565099.707, -13612896..."
2,3,2024-25,0604740,0161143,01611430000000,Alameda,Berkeley Unified,Unified,PK,12,...,0,0.0,1111,12.1,2686,29.3,10.434329,4.364648e+07,43695.341538,"POLYGON ((-13609482.48 4565074.597, -13609483...."
3,4,2024-25,0607800,0161150,01611500000000,Alameda,Castro Valley Unified,Unified,PK,12,...,0,0.0,1113,11.6,3728,39.0,66.885571,2.838285e+08,142492.767565,"MULTIPOLYGON (((-13582508.535 4529067.071, -13..."
4,5,2024-25,0612630,0161168,01611680000000,Alameda,Emery Unified,Unified,PK,12,...,0,0.0,81,13.7,412,69.8,1.273929,5.363392e+06,13741.272894,"POLYGON ((-13613999.038 4555592.769, -13614126..."


In [119]:
districts["DistrictTy"].value_counts()

DistrictTy
Elementary    516
Unified       345
High           76
Name: count, dtype: int64

In [120]:
print(districts.crs)

EPSG:3857


### Coordinate Reference System Alignment

The school district shapefile uses EPSG:3857, while the housing data stores property locations as latitude and longitude coordinates in EPSG:4326. Before performing a spatial join, I converted the school district boundaries to EPSG:4326 so that both datasets use the same coordinate reference system.

In [121]:
districts = districts.to_crs("EPSG:4326")

print("District CRS:", districts.crs)

District CRS: EPSG:4326


### Property Point Geometry

Each property was converted into a geographic point using its longitude and latitude. Longitude was used as the x-coordinate and latitude as the y-coordinate. The resulting GeoDataFrame was assigned EPSG:4326 to match the school district boundary data.

In [122]:
import geopandas as gpd

housing_gdf = gpd.GeoDataFrame(
    df.copy(),
    geometry=gpd.points_from_xy(
        df["Longitude"],
        df["Latitude"]
    ),
    crs="EPSG:4326"
)

print("Housing CRS:", housing_gdf.crs)
print("Housing rows:", len(housing_gdf))

housing_gdf[
    ["Latitude", "Longitude", "geometry"]
].head()

Housing CRS: EPSG:4326
Housing rows: 71279


,Latitude,Longitude,geometry
0,34.444024,-117.192685,POINT (-117.19268 34.44402)
1,33.741472,-117.787080,POINT (-117.78708 33.74147)
2,37.275959,-121.910612,POINT (-121.91061 37.27596)
3,37.990815,-121.770866,POINT (-121.77087 37.99082)
4,33.955252,-118.445999,POINT (-118.446 33.95525)


### Coordinate Quality Check

Only properties with valid California coordinates can be matched to a school district boundary. I checked whether latitude and longitude values were missing or outside a reasonable California range. Properties with invalid coordinates were retained in the dataset, but their school district values may remain missing after the spatial join.

In [123]:
valid_coordinates = (
    df["Latitude"].between(32, 42) &
    df["Longitude"].between(-125, -114)
)

print("Total properties:", len(df))
print("Valid coordinate rows:", valid_coordinates.sum())
print("Invalid or missing coordinate rows:", (~valid_coordinates).sum())
print(
    "Valid coordinate percentage:",
    round(valid_coordinates.mean() * 100, 2),
    "%"
)

Total properties: 71279
Valid coordinate rows: 71279
Invalid or missing coordinate rows: 0
Valid coordinate percentage: 100.0 %


### School District Feature Selection

The shapefile contains 54 columns, including administrative identifiers and demographic counts. For this analysis, I retained only the district name, district type, and geometry. DistrictName provides a detailed geographic category, while DistrictTY identifies whether the boundary represents an Elementary, High, or Unified district.

In [124]:
district_info = districts[
    [
        "DistrictNa",
        "DistrictTy",
        "geometry"
    ]
].copy()

district_info.head()

,DistrictNa,DistrictTy,geometry
0,Alameda Unified,Unified,"MULTIPOLYGON (((-122.22678 37.72651, -122.2267..."
1,Albany City Unified,Unified,"POLYGON ((-122.28671 37.89852, -122.28673 37.8..."
2,Berkeley Unified,Unified,"POLYGON ((-122.25606 37.89834, -122.25607 37.8..."
3,Castro Valley Unified,Unified,"MULTIPOLYGON (((-122.01375 37.64265, -122.0114..."
4,Emery Unified,Unified,"POLYGON ((-122.29663 37.8311, -122.29778 37.83..."


### Spatial Join

I spatially joined each property point to the California school district polygons using the `within` predicate. This assigns the district name and district type of every polygon that contains the property's coordinates.

In [125]:
housing_with_district = gpd.sjoin(
    housing_gdf,
    district_info,
    how="left",
    predicate="within"
)

### Multiple District Matches

California contains Elementary, High, and Unified school district boundaries. A property located in a Unified district may match one polygon, while a property in a separate Elementary and High district system may match two polygons. Therefore, I checked whether the spatial join created multiple rows for the same property before preparing the data for modeling.

In [126]:
print("Original housing rows:", len(housing_gdf))
print("Rows after spatial join:", len(housing_with_district))

match_count = (
    housing_with_district
    .groupby(level=0)
    .size()
)

print("\nNumber of district matches per property:")
print(match_count.value_counts().sort_index())

print(
    "Properties with multiple matches:",
    (match_count > 1).sum()
)

Original housing rows: 71279
Rows after spatial join: 88739

Number of district matches per property:
1    53819
2    17460
Name: count, dtype: int64
Properties with multiple matches: 17460


In [127]:
housing_with_district["DistrictTy"].value_counts()

DistrictTy
Unified       54084
Elementary    17507
High          17142
Name: count, dtype: int64

In [128]:
district_features = (
    housing_with_district
    .reset_index()
    .pivot_table(
        index="index",
        columns="DistrictTy",
        values="DistrictNa",
        aggfunc="first"
    )
)

district_features

DistrictTy,Elementary,High,Unified
index,,,
0,NaN,NaN,Apple Valley Unified
1,NaN,NaN,Tustin Unified
2,NaN,NaN,San Jose Unified
3,NaN,NaN,Antioch Unified
4,NaN,NaN,Los Angeles Unified
...,...,...,...
71274,NaN,NaN,Palm Springs Unified
71275,Fullerton Elementary,Fullerton Joint Union High,NaN
71276,NaN,NaN,Pomona Unified


### Handling Multiple School District Matches

Some California properties belong to separate Elementary and High school districts, while others belong to a Unified district. As a result, a spatial join may produce multiple rows for a single property.

Instead of discarding any matches, I reshaped the joined data so that each property remained as a single observation with separate columns for Elementary, High, and Unified school districts. This preserves all available district information while maintaining one row per property for model training.

In [129]:
district_features = district_features.rename(columns={
    "Elementary": "ElementaryDistrict",
    "High": "HighDistrict",
    "Unified": "UnifiedDistrict"
})

In [130]:
df = df.join(district_features)

In [131]:
print(df.shape)

df[
    [
        "ElementaryDistrict",
        "HighDistrict",
        "UnifiedDistrict"
    ]
].head(10)

(71279, 990)


,ElementaryDistrict,HighDistrict,UnifiedDistrict
0,NaN,NaN,Apple Valley Unified
1,NaN,NaN,Tustin Unified
2,NaN,NaN,San Jose Unified
3,NaN,NaN,Antioch Unified
4,NaN,NaN,Los Angeles Unified
5,NaN,NaN,Orange Unified
6,NaN,NaN,Poway Unified
7,NaN,NaN,Corona-Norco Unified
8,Huntington Beach City Elementary,Huntington Beach Union High,NaN
9,NaN,NaN,San Diego Unified


In [132]:
district_summary = pd.DataFrame({
    "Missing": df[
        [
            "ElementaryDistrict",
            "HighDistrict",
            "UnifiedDistrict"
        ]
    ].isna().sum(),
    "Missing %": (
        df[
            [
                "ElementaryDistrict",
                "HighDistrict",
                "UnifiedDistrict"
            ]
        ].isna().mean() * 100
    ).round(2),
    "Unique": df[
        [
            "ElementaryDistrict",
            "HighDistrict",
            "UnifiedDistrict"
        ]
    ].nunique()
})

district_summary

,Missing,Missing %,Unique
ElementaryDistrict,53772,75.44,315
HighDistrict,54137,75.95,73
UnifiedDistrict,17195,24.12,310


## Create additional engineered features

### 1. Property Age

A property's age may influence its market value because newer homes often require less maintenance and may include more modern layouts and building materials. I calculated the property age as the difference between the closing year and the construction year.

In [133]:
print(df.columns.tolist())

['ViewYN', 'PoolPrivateYN', 'Latitude', 'Longitude', 'LivingArea', 'DaysOnMarket', 'AttachedGarageYN', 'ParkingTotal', 'YearBuilt', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN', 'Stories', 'MainLevelBedrooms', 'NewConstructionYN', 'GarageSpaces', 'AssociationFee', 'LotSizeSquareFeet', 'MLSAreaMajor_101 - North Inglewood', 'MLSAreaMajor_102 - South Inglewood', 'MLSAreaMajor_103 - Ladera Heights', 'MLSAreaMajor_105 - Lennox', 'MLSAreaMajor_106 - Los Angeles', 'MLSAreaMajor_107 - Holly Glen/Del Aire', 'MLSAreaMajor_108 - North Hawthorne', 'MLSAreaMajor_109 - Ramona/Burleigh', 'MLSAreaMajor_11 - Westside', 'MLSAreaMajor_110 - East Hawthorne', 'MLSAreaMajor_111 - Bodger Park/El Camino', 'MLSAreaMajor_112 - North Lawndale', 'MLSAreaMajor_113 - South Lawndale', 'MLSAreaMajor_114 - Hollypark', 'MLSAreaMajor_115 - North Gardena', 'MLSAreaMajor_116 - North Gateway', 'MLSAreaMajor_117 - McCarthy', 'MLSAreaMajor_118 - Pacific Square', 'MLSAreaMajor_119 - Central Gardena', 'MLSAreaMajor_

### 2.HasHOA

This feature indicates whether a property belongs to a homeowners association (HOA). HOA communities often provide shared amenities and maintenance services, which may influence housing prices.

In [134]:
df["HasHOA"] = (
    df["AssociationFee"].fillna(0) > 0
).astype(int)

/var/folders/tf/71q912hd3fzd4yz1qccgzpcw0000gn/T/ipykernel_77196/2181925601.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["HasHOA"] = (


### 3.LivingAreaPerBedroom
Measures the average interior living space available for each bedroom

In [135]:
df["LivingAreaPerBedroom"] = np.where(
    df["BedroomsTotal"] > 0,
    df["LivingArea"] / df["BedroomsTotal"],
    np.nan
)

/var/folders/tf/71q912hd3fzd4yz1qccgzpcw0000gn/T/ipykernel_77196/531557342.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["LivingAreaPerBedroom"] = np.where(


### 4.LivingToLotRatio

Represents how much of the lot is occupied by living space, reflecting development density.

In [136]:
df["LivingToLotRatio"] = np.where(
    df["LotSizeSquareFeet"] > 0,
    df["LivingArea"] / df["LotSizeSquareFeet"],
    np.nan
)

/var/folders/tf/71q912hd3fzd4yz1qccgzpcw0000gn/T/ipykernel_77196/953094549.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["LivingToLotRatio"] = np.where(


### 5.BathroomsPerBedroom

In [137]:
df["BathroomsPerBedroom"] = np.where(
    df["BedroomsTotal"] > 0,
    df["BathroomsTotalInteger"] / df["BedroomsTotal"],
    np.nan
)

/var/folders/tf/71q912hd3fzd4yz1qccgzpcw0000gn/T/ipykernel_77196/992177979.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["BathroomsPerBedroom"] = np.where(


### save data with different features combination

In [138]:
from pathlib import Path

# Create output folder for different feature-engineering versions
OUTPUT_DIR = Path("../Week6/data_versions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Newly engineered features
school_district_features = [
    "ElementaryDistrict",
    "HighDistrict",
    "UnifiedDistrict"
]

hoa_features = [
    "AssociationFee"
]

ratio_features = [
    "LivingToLotRatio"
]

garage_features = [
    "HasGarage"
]

# Combine all engineered features
all_new_features = (
    school_district_features
    + hoa_features
    + ratio_features
    + garage_features
)

# Define different feature combinations for model comparison
feature_versions = {

    # Baseline feature set (same as Week 5)
    "v0_week5_baseline": [],

    # School district features only
    "v1_school_district": school_district_features,

    # HOA feature only
    "v2_hoa": hoa_features,

    # Ratio feature only
    "v3_ratio": ratio_features,

    # School district + HOA
    "v4_school_hoa":
        school_district_features
        + hoa_features,

    # School district + ratio
    "v5_school_ratio":
        school_district_features
        + ratio_features,

    # All engineered features
    "v6_all_features":
        all_new_features
}

saved_versions = []

for version_name, included_features in feature_versions.items():

    # Remove engineered features that are not included
    columns_to_drop = [
        col for col in all_new_features
        if col not in included_features
        and col in df.columns
    ]

    version_df = df.drop(columns=columns_to_drop).copy()

    output_path = OUTPUT_DIR / f"{version_name}.csv.gz"

    version_df.to_csv(
        output_path,
        index=False,
        compression="gzip"
    )

    saved_versions.append({
        "Version": version_name,
        "Rows": version_df.shape[0],
        "Columns": version_df.shape[1],
        "Engineered Features":
            ", ".join(included_features)
            if included_features else "None",
        "File": output_path.name
    })

    print(
        f"Saved {version_name}: "
        f"{version_df.shape[0]:,} rows × "
        f"{version_df.shape[1]:,} columns"
    )

# Save a summary of all exported datasets
version_summary = pd.DataFrame(saved_versions)

version_summary.to_csv(
    OUTPUT_DIR / "version_summary.csv",
    index=False
)

version_summary

Saved v0_week5_baseline: 71,279 rows × 989 columns
Saved v1_school_district: 71,279 rows × 992 columns
Saved v2_hoa: 71,279 rows × 990 columns
Saved v3_ratio: 71,279 rows × 990 columns
Saved v4_school_hoa: 71,279 rows × 993 columns
Saved v5_school_ratio: 71,279 rows × 993 columns
Saved v6_all_features: 71,279 rows × 994 columns


,Version,Rows,Columns,Engineered Features,File
0,v0_week5_baseline,71279,989,None,v0_week5_baseline.csv.gz
1,v1_school_district,71279,992,"ElementaryDistrict, HighDistrict, UnifiedDistrict",v1_school_district.csv.gz
2,v2_hoa,71279,990,AssociationFee,v2_hoa.csv.gz
3,v3_ratio,71279,990,LivingToLotRatio,v3_ratio.csv.gz
4,v4_school_hoa,71279,993,"ElementaryDistrict, HighDistrict, UnifiedDistr...",v4_school_hoa.csv.gz
5,v5_school_ratio,71279,993,"ElementaryDistrict, HighDistrict, UnifiedDistr...",v5_school_ratio.csv.gz
6,v6_all_features,71279,994,"ElementaryDistrict, HighDistrict, UnifiedDistr...",v6_all_features.csv.gz
